# thaw GPU Benchmark — Pipelined Restore

Tests the Rust+CUDA hot path: double-buffered async DMA with O_DIRECT.

**Setup:** Colab Pro with GPU runtime (tested on RTX PRO 6000 Blackwell).

**Before running:** Add secrets in Colab sidebar:
- `GITHUB_PAT` — GitHub personal access token
- `HF_TOKEN` — HuggingFace token (for gated models like Llama-3)

In [ ]:
# [1] GPU info
!nvidia-smi --query-gpu=name,memory.total,pcie.link.gen.current,pcie.link.width.current --format=csv,noheader

In [ ]:
# [2] Install Rust + setup env
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable 2>&1 | tail -3
import os
from google.colab import userdata
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!huggingface-cli login --token $HF_TOKEN
!huggingface-cli whoami
!rustc --version

In [ ]:
# [3] Clone repo
!git clone https://github.com/thaw-ai/thaw.git /content/thaw 2>/dev/null || (cd /content/thaw && git pull)
!cd /content/thaw && git log --oneline -3

In [ ]:
# [4] Rust tests (mock backend, no GPU needed)
!cd /content/thaw && cargo test -p thaw-core -p thaw-runtime 2>&1 | tail -10

In [ ]:
# [5] Rust tests with real CUDA
!cd /content/thaw && CUDA_PATH=/usr/local/cuda cargo test --release -p thaw-runtime --features cuda 2>&1 | tail -10

In [ ]:
# [6] Build benchmark binary
!cd /content/thaw && CUDA_PATH=/usr/local/cuda cargo build --release -p thaw-cli --features cuda 2>&1 | tail -3

In [ ]:
# [7] Benchmark: 1 GB
!cd /content/thaw && CUDA_PATH=/usr/local/cuda cargo run --release -p thaw-cli --features cuda -- 1024

In [ ]:
# [8] Benchmark: 4 GB
!cd /content/thaw && CUDA_PATH=/usr/local/cuda cargo run --release -p thaw-cli --features cuda -- 4096

In [ ]:
# [9] Benchmark: 8 GB (Llama-3-8B scale)
!cd /content/thaw && CUDA_PATH=/usr/local/cuda cargo run --release -p thaw-cli --features cuda -- 8192

---
## Python Integration

In [ ]:
# [10] Build Python extension (PyO3)
!pip install "maturin[patchelf]" -q
!cd /content/thaw && CUDA_PATH=/usr/local/cuda maturin build --release -m crates/thaw-py/Cargo.toml --features cuda --skip-auditwheel 2>&1 | tail -3
!pip install /content/thaw/target/wheels/*.whl --force-reinstall -q
import thaw
print('thaw loaded:', [x for x in dir(thaw) if not x.startswith('_')])

In [ ]:
# [11] Fix vLLM fileno() crash in Jupyter — run before vLLM cell
import sys
sys.stdout._original_stdstream_copy = 1
sys.stderr._original_stdstream_copy = 2
print("fileno patch applied")

In [ ]:
# [12] vLLM Llama-3-8B cold start: normal vs thaw
!pip install vllm hf_transfer -q 2>&1 | tail -3

# Use Rust-based downloader (fixes HF CDN stalling)
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!HF_HUB_ENABLE_HF_TRANSFER=1 huggingface-cli download meta-llama/Meta-Llama-3-8B --exclude "original/*"

import sys, gc, time
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
sys.path.insert(0, '/content/thaw/python')

# Fix vLLM fileno() crash in Jupyter
sys.stdout._original_stdstream_copy = 1
sys.stderr._original_stdstream_copy = 2

import torch
from vllm import LLM, SamplingParams
from thaw_vllm import freeze_model_pipelined, restore_model_pipelined

MODEL = "meta-llama/Meta-Llama-3-8B"
SNAPSHOT = "/tmp/llama3-8b.thaw"
prompt = "The future of artificial intelligence is"
sampling = SamplingParams(temperature=0, max_tokens=50)

def find_model(llm):
    """Extract the nn.Module from a vLLM LLM instance."""
    engine = llm.llm_engine
    for path_fn in [
        lambda: engine.model_executor.driver_worker.model_runner.model,
        lambda: engine.engine_core.model_runner.model,
        lambda: engine.engine_core.model_executor.driver_worker.model_runner.model,
        lambda: engine.engine_core.model_executor.model_runner.model,
        lambda: engine.get_model(),
    ]:
        try:
            return path_fn()
        except (AttributeError, TypeError):
            continue
    raise RuntimeError("Could not find nn.Module in vLLM")

# === Phase 1: Normal cold start ===
print("=" * 60)
print("Phase 1: Normal vLLM cold start")
print("=" * 60)
t0 = time.perf_counter()
llm = LLM(model=MODEL, dtype="float16", enforce_eager=True,
          tensor_parallel_size=1, gpu_memory_utilization=0.4)
normal_time = time.perf_counter() - t0
print(f"Normal load: {normal_time:.1f}s")
out = llm.generate([prompt], sampling)
ref_text = out[0].outputs[0].text
print(f"Output: {ref_text}")

# === Phase 2: Freeze (pipelined) ===
print("\n" + "=" * 60)
print("Phase 2: Freeze to .thaw snapshot (pipelined)")
print("=" * 60)
model = find_model(llm)
stats = freeze_model_pipelined(model, SNAPSHOT)
print(f"Frozen: {stats['num_regions']} regions, {stats['total_bytes']/1e9:.2f} GB")
print(f"Freeze: {stats['elapsed_s']:.1f}s ({stats['throughput_gb_s']:.2f} GB/s) [{stats.get('backend', 'python')}]")
del llm, model; gc.collect(); torch.cuda.empty_cache()

# === Phase 3: Thaw cold start ===
print("\n" + "=" * 60)
print("Phase 3: Thaw-powered cold start")
print("=" * 60)
t0 = time.perf_counter()
llm_thaw = LLM(model=MODEL, dtype="float16", enforce_eager=True,
               tensor_parallel_size=1, gpu_memory_utilization=0.4,
               load_format="dummy")
init_time = time.perf_counter() - t0
print(f"Dummy init: {init_time:.1f}s")

t0 = time.perf_counter()
model = find_model(llm_thaw)
rstats = restore_model_pipelined(model, SNAPSHOT)
restore_time = time.perf_counter() - t0
backend = rstats.get("backend", "python")
thaw_total = init_time + restore_time
print(f"Restore: {restore_time:.1f}s ({rstats['throughput_gb_s']:.2f} GB/s) [{backend}]")
print(f"Total thaw cold start: {thaw_total:.1f}s")

out = llm_thaw.generate([prompt], sampling)
thaw_text = out[0].outputs[0].text
print(f"Output: {thaw_text}")

# === Results ===
print("\n" + "=" * 60)
print("RESULTS")
print("=" * 60)
print(f"  Normal vLLM cold start:  {normal_time:.1f}s")
print(f"  Thaw cold start:         {thaw_total:.1f}s  [{backend}]")
print(f"    - dummy init:          {init_time:.1f}s")
print(f"    - weight restore:      {restore_time:.1f}s ({rstats['throughput_gb_s']:.2f} GB/s)")
print(f"  Speedup:                 {normal_time / thaw_total:.1f}x")
print(f"  Freeze:                  {stats['elapsed_s']:.1f}s ({stats['throughput_gb_s']:.2f} GB/s) [{stats.get('backend', 'python')}]")
match = ref_text == thaw_text
print(f"  Output match:            {'PASS' if match else 'FAIL'}")
if not match:
    print(f"    Expected: {ref_text[:100]}")
    print(f"    Got:      {thaw_text[:100]}")